
  <img src="https://github.com/gefero/factor_data_tuto_NLP_SICSS/blob/main/imgs/logo_final_conjunto.png?raw=true" width="80%">


# Summer Institute in Computational Social Sciences - Buenos Aires 2026
# Taller: Procesamiento de Lenguaje Natural y polarización
# Anotación y LLMs

### Profesor: Juan Manuel Pérez




En esta notebook vamos a jugar un poco con la API de Gemini.

Regístrense en [la página de Gemini/Google AI Studio y generen una API KEY](https://ai.google.dev/gemini-api/docs/api-key)

Una vez que la generen, la ponen acá (no la copypasteo por obvias razones)

In [ ]:
!pip install google-genai datasets -qqq

In [ ]:
import getpass
from google import genai
from google.genai import types
from google.colab import userdata

In [ ]:
client = genai.Client(api_key=userdata.get("gemini-test"))

## Zero-shot classification

Podemos usar un LLM para que haga tareas para las cuales no fue entrenado. Por ejemplo, podemos usarlo para clasificar textos.

Para eso, usaremos un prompt que le diga qué hacer. Por ejemplo, si le decimos "Clasificá este texto como positivo o negativo", y le damos un texto, nos va a decir si es positivo o negativo.

In [ ]:

text = """Clasificar el siguiente texto en positivo, negativo, o neutral. Pensá paso a paso la respuesta y contestá 'La respuesta final es' y el sentimiento (uno de los siguientes: positivo, negativo, neutral) al final de la respuesta. No respondas nada más que eso.

texto: MESSI SOS EL GOAT LPM"""

model_name = "gemini-3.5-flash"
response = client.models.generate_content(
    model=model_name, contents=text
)
print(response.text)


Para clasificar el texto, analizamos sus componentes paso a paso:

1. **"MESSI"**: Se refiere al futbolista Lionel Messi, el sujeto del mensaje.
2. **"SOS EL GOAT"**: "GOAT" es el acrónimo en inglés de *Greatest Of All Time* (El mejor de todos los tiempos). Decirle a alguien que es el "GOAT" es uno de los mayores elogios posibles en el ámbito deportivo, lo que denota una fuerte carga positiva.
3. **"LPM"**: Es la abreviatura de "La puta madre". Aunque literalmente es un insulto/maldición, en el contexto de la cultura rioplatense (Argentina/Uruguay) y combinado con "sos el GOAT", se utiliza como una interjección de euforia, asombro, emoción extrema y admiración. No tiene una connotación negativa en este caso, sino que intensifica la positividad del elogio.

Al evaluar el mensaje en su conjunto, se trata de una expresión de profunda admiración, alegría y entusiasmo.

La respuesta final es positivo


## Json mode

In [ ]:
from typing import Literal
from enum import Enum
from pydantic import BaseModel, Field

class HateCategories(str, Enum):
    WOMEN = "WOMEN"
    LGBTI = "LGBTI"
    RACISM = "RACISM"
    CLASS = "CLASS"
    POLITICS = "POLITICS"
    DISABLED = "DISABLED"
    APPEARANCE = "APPEARANCE"
    CRIMINAL = "CRIMINAL"

class CommentLabel(BaseModel):
    HATEFUL: bool = Field(
        description="El comentario contiene discurso de odio explícito hacia un grupo."
    )
    OFFENSIVE: bool = Field(
        description=(
            "El comentario contiene lenguaje ofensivo, insultos o agresividad, "
            "independientemente de si es discurso de odio."
        )
    )
    CALLS: bool = Field(
        description="Incita a actuar en contra de alguien. Solo aplica si HATEFUL=True."
    )
    categories: list[HateCategories]


TEMPLATE_PROMPT = """Tu tarea es detectar discurso de odio en comentarios de Twitter a noticias periodísticas.

Definiciones:
- HATEFUL: discurso de odio explícito hacia un grupo (por género, raza, orientación sexual, etc.)
- OFFENSIVE: lenguaje ofensivo, insultos o agresividad, independientemente de si es discurso de odio.
- CALLS: incitación a actuar en contra de alguien (solo si HATEFUL=True)
- Las categorías WOMEN/LGBTI/RACISM/CLASS/POLITICS/DISABLED/APPEARANCE/CRIMINAL
  solo aplican si HATEFUL=True; marcalas en False si HATEFUL=False.

Definiciones de las categorías:
- WOMEN: Sexismo o misoginia
- LGBTI: Homofobia o transfobia
- RACISM: Racismo o xenofobia
- CLASS: Discriminación por clase social
- POLITICS: Odio por afiliación política
- DISABLED: Discriminación por discapacidad
- APPEARANCE: Ataque a la apariencia física
- CRIMINAL: discurso de odio contra personas privadas de libertad

Responder con un JSON con los siguientes campos:
{{
    "HATEFUL": bool,
    "OFFENSIVE": bool,
    "CALLS": bool,
    "categories": list[str]
}}

---

Contexto: {contexto}
Comentario: {comentario}
"""
contexto = '''Repudian una serie de declaraciones de Eduardo Feinmann sobre los chinos: "El planeta se va a ocupar de ustedes"'''
comentario = '''@usuario Todos los políticamente correctos se dan cuenta que hay una pandemia por culpa de los chinos?hay que hacer algo con esta gente urgente...'''

prompt = TEMPLATE_PROMPT.format(contexto=contexto, comentario=comentario)
response = client.models.generate_content(
    model=model_name, contents=prompt,
    config=types.GenerateContentConfig(
        response_mime_type='application/json',
        response_schema=CommentLabel,
    ),
)

# Probar agregar CoT

print(response.text)

{"HATEFUL":true,"OFFENSIVE":true,"CALLS":true,"categories":["RACISM"]}


In [ ]:
response.parsed

CommentLabel(HATEFUL=True, OFFENSIVE=True, CALLS=True, categories=[<HateCategories.RACISM: 'RACISM'>])

In [ ]:
response = client.models.generate_content(
  model="gemini-3.5-flash",
  contents=prompt,
  config=types.GenerateContentConfig(
    response_mime_type='application/json',
    response_schema=CommentLabel,
    thinking_config=types.ThinkingConfig(
      include_thoughts=True
    )
  )
)

In [ ]:
response.candidates[0].content.parts

[Part(
   text="""**My Analysis of This Tweet**
 
 Okay, let's break this down. My goal is to classify this tweet according to the hate speech detection schema. The context is a repudiation of Eduardo Feinmann's statements about the Chinese, which is relevant, but the real meat is in the user's comment.
 
 The comment itself is pretty blatant. I see two main red flags right away. First, it directly blames "the Chinese" for the pandemic, which is classic xenophobia. Secondly, the phrase "hay que hacer algo con esta gente urgente..."—"we must do something about these people urgently"—is an implicit call to action, and a pretty sinister one at that. It's not subtle.
 
 So, let's go category by category. Is it *HATEFUL*? Absolutely. The language targets an entire group based on their national origin with blame and a clear suggestion for action, implying harm. *OFFENSIVE*? Definitely. The comment is aggressive, hostile, and clearly aimed at denigrating a group. Finally, does it contain *CAL

In [ ]:
!curl https://raw.githubusercontent.com/finiteautomata/sicss/refs/heads/main/data/sample.json -o sample.json # nos bajamos los datitos

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  470k  100  470k    0     0  3415k      0 --:--:-- --:--:-- --:--:-- 3432k


In [ ]:
import json

data = []
for line in open("sample.json"):
    data.append(json.loads(line.strip()))

In [ ]:
from tqdm import tqdm
import time

for example in tqdm(data):
    if "response" in example:
        continue
    contexto = example["context_tweet"]
    text = example["text"]
    prompt = TEMPLATE_PROMPT.format(contexto=contexto, comentario=text)

    response = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type='application/json',
            response_schema=CommentLabel,
            thinking_config=types.ThinkingConfig(
            include_thoughts=True
            )
        )
    )

    example["response"] = response

    time.sleep(5)


100%|██████████| 74/74 [07:51<00:00,  6.37s/it]


In [ ]:
example

{'text': '@usuario Y que tiene de malo? Bien muerto el negro de mierda lastima que se gastaron 2 balas, ya cuando entran a la cárcel hay que matarlos, nos salen caros y los mantenemos con los impuestos',
 'article_id': '1286038489646215170',
 'annotators': ['annotator_2', 'annotator_1', 'annotator_5'],
 'HATEFUL': ['annotator_2', 'annotator_1', 'annotator_5'],
 'CALLS': ['annotator_5'],
 'WOMEN': [],
 'LGBTI': [],
 'RACISM': ['annotator_2', 'annotator_1', 'annotator_5'],
 'CLASS': [],
 'POLITICS': [],
 'DISABLED': [],
 'APPEARANCE': [],
 'CRIMINAL': ['annotator_2', 'annotator_5'],
 'tweet_id': '1286128306765606912',
 'user_id': '1276163056406540296',
 'id': 375473,
 'is_hateful': True,
 'categories': ['RACISM', 'CRIMINAL'],
 'calls_to_action': True,
 'min_label': 2,
 'context_tweet': 'La declaración de un testigo del caso del jubilado que mató a un ladrón: “Levantaba la mano pidiendo ayuda y el hombre le volvió a disparar” | Por Leonardo Scannone https://t.co/46a0CqmsaD',
 'title': 'La